# **Prepration**

In [2]:
import pandas as pd
import numpy as np
import torch
import json
import shap
import re
from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline
from tqdm import tqdm
import nltk
from nltk.tokenize import sent_tokenize
from nltk.corpus import stopwords
#import torch
#import scipy as sp

In [ ]:
model_path = '/XLM-R-base'
test_data_path = '/testing.csv'
output_path = "/shap_word_analysis_xlm-r.json"

model = AutoModelForSequenceClassification.from_pretrained(model_path)
tokenizer = AutoTokenizer.from_pretrained(model_path)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

XLMRobertaForSequenceClassification(
  (classifier): XLMRobertaClassificationHead(
    (dense): Linear(in_features=768, out_features=768, bias=True)
    (dropout): Dropout(p=0.1, inplace=False)
    (out_proj): Linear(in_features=768, out_features=3, bias=True)
  )
  (roberta): XLMRobertaModel(
    (embeddings): XLMRobertaEmbeddings(
      (word_embeddings): Embedding(250002, 768, padding_idx=1)
      (token_type_embeddings): Embedding(1, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
      (position_embeddings): Embedding(514, 768, padding_idx=1)
    )
    (encoder): XLMRobertaEncoder(
      (layer): ModuleList(
        (0-11): 12 x XLMRobertaLayer(
          (attention): XLMRobertaAttention(
            (self): XLMRobertaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Li

In [4]:
df = pd.read_csv(test_data_path)

In [ ]:
def truncate_to_max_tokens(text, tokenizer, max_length=512):
    text = str(text)
    enc = tokenizer(text, return_offsets_mapping=True, add_special_tokens=False)
    if len(enc['input_ids']) <= max_length - 2:   # room for <s> and </s>
        return text
    char_end = enc['offset_mapping'][max_length - 3][1]
    return text[:char_end]

df['text'] = df['text'].apply(lambda x: truncate_to_max_tokens(x, tokenizer, max_length=512))

# **Pipeline**

In [6]:
# custom SHAP function
def g(texts):
    if not isinstance(texts, list):
        texts = list(texts) if isinstance(texts, np.ndarray) else [texts]

    texts = ["" if x is None else str(x) for x in texts]

    encoder = tokenizer(texts, padding=True, truncation=True, max_length=512, return_tensors="pt")
    encoder = {k: v.to(device) for k, v in encoder.items()}
    model.eval()
    with torch.no_grad():
        logits = model(**encoder).logits.detach().cpu().numpy()
    return logits

In [7]:
masker = shap.maskers.Text(tokenizer)
#n_classes = model.config.num_labels if hasattr(model.config, "num_labels") else None
#output_names = [str(i) for i in range(n_classes)] if n_classes is not None else None
explainer = shap.Explainer(g, masker, batch_size=128)

# Max_evals Investigation

In [8]:
# Check SHAP's default

import inspect
sig = inspect.signature(explainer.__call__)
print(f"Default max_evals parameter: {sig.parameters['max_evals'].default}")

Default max_evals parameter: 500


In [ ]:
'''
import time
import random

# Random sample of 30
all_texts = df['text'].tolist()
sample_indices = random.sample(range(len(all_texts)), 30)
test_texts = [all_texts[i] for i in sample_indices]

# Check actual token counts of the sample
sample_tokens = [len(tokenizer.encode(str(t))) for t in test_texts]
print(f"Sample token stats — Mean: {np.mean(sample_tokens):.0f}, Min: {min(sample_tokens)}, Max: {max(sample_tokens)}")

# With ~300 tokens, one pass = 600 evals
# 8 passes = 4800, but stability may come earlier
eval_levels = [1500, 2000, 2500, 3000, 3500]
all_results = {}

for me in eval_levels:
    start = time.time()
    shap_out = explainer(test_texts, max_evals=me, silent=True)
    elapsed = time.time() - start

    values = []
    for j in range(len(test_texts)):
        logits = g([test_texts[j]])
        pred = int(np.argmax(logits[0]))
        values.append(shap_out[j].values[:, pred])

    all_results[me] = values
    per_sample = elapsed / len(test_texts)
    print(f"max_evals={me}: {elapsed:.1f}s total | {per_sample:.1f}s per sample | projected full: {per_sample * len(df) / 3600:.1f} hrs")

print("\nStability analysis:")
for i in range(1, len(eval_levels)):
    prev = eval_levels[i-1]
    curr = eval_levels[i]
    correlations = []
    for j in range(len(test_texts)):
        corr = np.corrcoef(all_results[prev][j], all_results[curr][j])[0, 1]
        correlations.append(corr)
    mean_corr = np.mean(correlations)
    print(f"  {prev} vs {curr}: mean correlation = {mean_corr:.4f} | {'✓ STABLE' if mean_corr >= 0.99 else ''}")
  '''

# **Analyze SHAP values**

In [10]:
def get_word_level_shap(tokens, shap_values_matrix, fix_sentencepiece=False):

    words = []
    word_shap_values = []

    group_tokens = []
    group_shaps = []

    for token, shap_val in zip(tokens, shap_values_matrix):
        if token == '':
            continue

        group_tokens.append(token.rstrip())
        group_shaps.append(shap_val)

        # Trailing space means end of word
        if token.endswith(' '):
            # Trim leading symbol-only tokens
            while group_tokens and not any(ch.isalnum() for ch in group_tokens[0]):
                group_tokens.pop(0)
                group_shaps.pop(0)

            # Trim trailing symbol-only tokens
            while group_tokens and not any(ch.isalnum() for ch in group_tokens[-1]):
                group_tokens.pop()
                group_shaps.pop()

            if group_tokens:
                word = merge_tokens(group_tokens, fix_sentencepiece)
                words.append(word)
                word_shap_values.append(np.sum(group_shaps, axis=0))

            group_tokens = []
            group_shaps = []

    # Last word
    if group_tokens:
        while group_tokens and not any(ch.isalnum() for ch in group_tokens[0]):
            group_tokens.pop(0)
            group_shaps.pop(0)
        while group_tokens and not any(ch.isalnum() for ch in group_tokens[-1]):
            group_tokens.pop()
            group_shaps.pop()
        if group_tokens:
            word = merge_tokens(group_tokens, fix_sentencepiece)
            words.append(word)
            word_shap_values.append(np.sum(group_shaps, axis=0))

    return words, word_shap_values


def merge_tokens(group_tokens, fix_sentencepiece=False):
    """
    Join subword tokens into a word, handling the SentencePiece duplication
    where a single char like 'F' is followed by 'Fraktion' (already contains 'F').
    Skip the standalone char from the string but its SHAP value is still summed.
    """

    if not fix_sentencepiece:
      return ''.join(group_tokens)


    merged = []
    for i, tok in enumerate(group_tokens):
        # Check: is this a single character followed by a token that starts with it?
        if (len(tok) == 1
            and i + 1 < len(group_tokens)
            and group_tokens[i + 1].startswith(tok)):
            # Skip this token in the string — it's a SentencePiece artifact
            continue
        merged.append(tok)

    return ''.join(merged)

In [11]:
def extract_shap_data(text, shap_exp, logits, fix_sentencepiece=False):

    predicted_class = int(np.argmax(logits))
    token_list = shap_exp.data
    token_shap_matrix = shap_exp.values          # shape (n_tokens, 3)
    base_values = shap_exp.base_values            # shape (3,)

    # softmax
    exp_logits = np.exp(logits - np.max(logits))
    probabilities = exp_logits / exp_logits.sum()

    # word-level aggregation across all 3 classes
    words, word_shap_list = get_word_level_shap(token_list, token_shap_matrix, fix_sentencepiece)

    return {
        'text': text,
        'predicted_class': predicted_class,
        'logits': [float(v) for v in logits],
        'probabilities': [float(v) for v in probabilities],
        'confidence_margin': float(sorted(probabilities)[-1] - sorted(probabilities)[-2]),
        'base_values': [float(v) for v in base_values],

        # raw token level
        'tokens': list(token_list),
        'token_count': len([t for t in token_list if t != '']),
        'token_shap_values': {
            'left':   [float(v) for v in token_shap_matrix[:, 0]],
            'center': [float(v) for v in token_shap_matrix[:, 1]],
            'right':  [float(v) for v in token_shap_matrix[:, 2]]
        },

        # word level
        'words': words,
        'word_count': len(words),
        'word_shap_values': {
            'left':   [float(v[0]) for v in word_shap_list],
            'center': [float(v[1]) for v in word_shap_list],
            'right':  [float(v[2]) for v in word_shap_list]
        }
    }

In [12]:
def process_all_samples(df, explainer, tokenizer, model, batch_size, fix_sentencepiece=False):

    results = []

    texts = df['text'].tolist()
    labels = df['label'].tolist()
    for i in tqdm(range(0, len(texts), batch_size), desc="Explaining batches"):
        batch_texts = texts[i:i+batch_size]
        batch_labels = labels[i:i+batch_size]

        # get SHAP explanations for the whole batch (tune max_evals for speed/precision)
        shap_batch = explainer(batch_texts, max_evals=2500, silent=True)

        # et logits once for the whole batch (reuse for predicted class)
        logits_batch = g(batch_texts)   # fast batch inference

        # process each sample in the batch
        for j, text in enumerate(batch_texts):
            idx = i + j
            try:
                shap_exp = shap_batch[j]
                sample_result = extract_shap_data(
                    text,
                    shap_exp,
                    logits=logits_batch[j],
                    fix_sentencepiece=fix_sentencepiece
                )

                sample_result['sample_index'] = idx
                sample_result['true_class'] = int(batch_labels[j])
                sample_result['is_correct'] = (sample_result['predicted_class'] == batch_labels[j])

                results.append(sample_result)
            except Exception as e:
                print(f"Error processing sample {idx}: {str(e)}")
                continue

    return results

In [ ]:
# Process all test samples
results = process_all_samples(df, explainer, tokenizer, model, batch_size=16, fix_sentencepiece=True)

with open(output_path, 'w', encoding='utf-8') as f:
    json.dump(results, f, ensure_ascii=False, indent=2)

print(f"\nTotal samples processed: {len(results)}")
print(f"Correct predictions: {sum(1 for r in results if r['is_correct'])}/{len(results)} "
      f"({sum(1 for r in results if r['is_correct'])/len(results)*100:.1f}%)")

for c in [0, 1, 2]:
    class_results = [r for r in results if r['predicted_class'] == c]
    class_name = ['Left', 'Center', 'Right'][c]
    if class_results:
        correct = sum(1 for r in class_results if r['is_correct'])
        print(f"\n  {class_name} (predicted): {len(class_results)} samples, "
              f"accuracy {correct/len(class_results)*100:.1f}%")